# Stage 9 — Base ASAG Scoring Pipeline

**Project:** Automatic Short Answer Grading using Semantic and Dependency Analysis  
**Repository:** `proposition-level-asag`  
**Current Stage:** Stage 9 — Base ASAG Scoring  

---

### Purpose
This notebook demonstrates the **Stage 9 Component** for the Base ASAG system.

In previous stages, we implemented the complete foundational NLP pipeline:
- **Stage 3:** Linguistic Preprocessing (tokenization, lemmatization, POS tagging).
- **Stage 4:** Dependency Parsing (syntactic heads, grammatical relations).
- **Stage 5:** Dependency Triples (canonical relational clause structures).
- **Stage 6:** Semantic & Lexical Relationships (exact, lemma, POS guard, WordNet path).
- **Stage 7:** Dependency Triple Matching (one-to-one greedy alignment, continuous alignment scores).
- **Stage 8:** Textual Entailment / Relevance (`ENTAILS`, `CONTRADICTS`, `UNCERTAIN`).

**Stage 9 computes a transparent, evidence-based Base ASAG score based on Reference Proposition Coverage.**

### Methodological Foundation (Basak et al., 2019 inspired)
In classical entailment-based ASAG, the grading task evaluates **how much of the reference answer's necessary propositional content is supported by the student's answer**.

Each reference proposition is evaluated into one of four states:
1. **`ENTAILED`:** Full credit ($1.0$).
2. **`UNCERTAIN`:** Configurable partial credit ($0.5$ by default).
3. **`CONTRADICTED`:** Zero credit ($0.0$).
4. **`MISSING`:** Zero credit ($0.0$).

The core research score is normalized strictly in $[0.0, 1.0]$:
$$\text{normalized\_score} = \frac{\sum_{i=1}^{N_{\text{ref}}} w_i \cdot \text{contribution}_i}{\sum_{i=1}^{N_{\text{ref}}} w_i}$$

> **CRITICAL RESEARCH BOUNDARY — UNMATCHED STUDENT INFORMATION:**  
> Base ASAG evaluates reference proposition coverage only. Any extra information written by the student that is not matched to the reference answer is:
> - Explicitly preserved in `unmatched_student_relations`
> - Reported in diagnostics
> - Awarded **neither a bonus nor a penalty** in Base ASAG ($0.0$ effect).
>
> It is **not** classified as correct, irrelevant, or incorrect in Stage 9. This creates the foundational empirical baseline that the Proposed Extension (Stages 11–14) will investigate.

> **NOTICE: DEMO DATA — NOT RESEARCH DATA**  
> Synthetic demonstration examples for pipeline verification only.

## 1. Imports and Setup

We import pandas and the full Base ASAG pipeline components up to Stage 9:
- Stage 4: `DependencyParser`
- Stage 5: `DependencyTripleExtractor`
- Stage 7: `TripleMatcher`
- Stage 8: `PropositionalEntailmentAnalyzer`
- Stage 9: `BaseASAGScorer`, `BaseScoreResult`, `PropositionScoreRecord`

In [1]:
import sys
from pathlib import Path
import pandas as pd

# Ensure repository root is on sys.path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from base_asag.dependency import DependencyParser, DependencyTripleExtractor
from base_asag.matching import TripleMatcher
from base_asag.entailment import PropositionalEntailmentAnalyzer
from base_asag.scoring import BaseASAGScorer, BaseScoreResult, PropositionScoreRecord

# Initialize pipeline modules
parser = DependencyParser(model_name="en_core_web_sm")
extractor = DependencyTripleExtractor()
matcher = TripleMatcher(alignment_threshold=0.5)
analyzer = PropositionalEntailmentAnalyzer(
    entailment_threshold=0.70,
    min_slot_similarity=0.50,
    contradiction_threshold=0.50,
)
scorer = BaseASAGScorer(
    entails_weight=1.0,
    uncertain_weight=0.5,
    contradicts_weight=0.0,
    missing_weight=0.0,
)

print("Initialized Base ASAG Scorer successfully.")
print(f"Weights: Entailed={scorer.entails_weight}, Uncertain={scorer.uncertain_weight}, Contradicted={scorer.contradicts_weight}, Missing={scorer.missing_weight}")

Initialized Base ASAG Scorer successfully.
Weights: Entailed=1.0, Uncertain=0.5, Contradicted=0.0, Missing=0.0


## 2. Helper Evaluation Function

To run the complete Base ASAG pipeline from raw text to Stage 9 score, we define a small helper function that connects Stages 4, 5, 7, 8, and 9.

In [2]:
def evaluate_and_score(ref_text: str, stu_text: str, custom_weights=None) -> BaseScoreResult:
    # 1. Parse and extract dependency structures (Stages 4 & 5)
    ref_doc = parser.parse(ref_text)
    stu_doc = parser.parse(stu_text)
    ref_triples = extractor.extract_document(ref_doc)
    stu_triples = extractor.extract_document(stu_doc)
    
    # 2. Match dependency triples (Stage 7)
    match_res = matcher.match(ref_triples, stu_triples)
    
    # 3. Evaluate proposition-level entailment (Stage 8)
    entail_list = analyzer.evaluate_matching_result(match_res, ref_doc, stu_doc)
    
    # 4. Compute Base ASAG score (Stage 9)
    return scorer.score_pipeline(match_res, entail_list, reference_weights=custom_weights)

print("Helper pipeline function defined.")

Helper pipeline function defined.


## 3. Demonstration 1: Fully Correct Answer

- **Reference Answer:** *"A primary key uniquely identifies each record in a table."*  
- **Student Answer:** *"A primary key uniquely identifies each record in a table."*  

The single reference proposition is entailed. Base score = $1.0$ ($5.0 / 5.0$).

In [3]:
res1 = evaluate_and_score(
    ref_text="A primary key uniquely identifies each record in a table.",
    stu_text="A primary key uniquely identifies each record in a table.",
)

print(f"Normalized Score: {res1.normalized_score} / 1.0")
print(f"Scaled Score:     {res1.scale_score(5.0)} / 5.0")
print(f"Letter Grade:     {res1.map_to_letter_grade()}")
print(f"Entailed: {res1.entailed_count}, Missing: {res1.missing_count}, Uncertain: {res1.uncertain_count}, Contradicted: {res1.contradicted_count}")

Normalized Score: 1.0 / 1.0
Scaled Score:     5.0 / 5.0
Letter Grade:     A
Entailed: 1, Missing: 0, Uncertain: 0, Contradicted: 0


## 4. Demonstration 2: Paraphrased Answer (`record` vs `row`)

- **Reference Answer:** *"A primary key uniquely identifies each record in a table."*  
- **Student Answer:** *"A primary key uniquely identifies every row in a table."*  

In Stage 6, WordNet path similarity between `record` and `row` is $0.125$.  
In Stage 8, this falls below the conservative slot threshold ($0.50$), producing `UNCERTAIN`.  
In Stage 9, the uncertain proposition receives the configured partial credit ($0.5$), yielding score $0.5$ ($2.5 / 5.0$).

In [4]:
res2 = evaluate_and_score(
    ref_text="A primary key uniquely identifies each record in a table.",
    stu_text="A primary key uniquely identifies every row in a table.",
)

print(f"Normalized Score: {res2.normalized_score} / 1.0")
print(f"Scaled Score:     {res2.scale_score(5.0)} / 5.0")
print(f"Letter Grade:     {res2.map_to_letter_grade()}")
print(f"Status:           {res2.proposition_results[0].status}")
print(f"Contribution:     {res2.proposition_results[0].contribution}")

Normalized Score: 0.5 / 1.0
Scaled Score:     2.5 / 5.0
Letter Grade:     F
Status:           UNCERTAIN
Contribution:     0.5


## 5. Demonstration 3: Partially Covered Answer (Missing Proposition)

- **Reference Answer:** *"A primary key uniquely identifies each record and cannot contain NULL values."*  
  - $R_1$: `(key, identify, record)`
  - $R_2$: `(key, contain, values)`
- **Student Answer:** *"A primary key uniquely identifies each record in a table."*  

The student covers $R_1$ but omits $R_2$.  
Base score = $(1.0 + 0.0) / 2 = 0.50$ ($2.5 / 5.0$).

In [5]:
res3 = evaluate_and_score(
    ref_text="A primary key uniquely identifies each record and cannot contain NULL values.",
    stu_text="A primary key uniquely identifies each record in a table.",
)

print(f"Normalized Score: {res3.normalized_score} / 1.0")
print(f"Scaled Score:     {res3.scale_score(5.0)} / 5.0")
print(f"Total Ref Propositions: {res3.total_reference_propositions}")
print(f"Entailed: {res3.entailed_count}, Missing: {res3.missing_count}\n")

for i, p in enumerate(res3.proposition_results):
    print(f"Ref R{i+1}: {p.reference_relation.to_tuple()} -> Status: {p.status}, Contribution: {p.contribution}")

Normalized Score: 0.5 / 1.0
Scaled Score:     2.5 / 5.0
Total Ref Propositions: 2
Entailed: 1, Missing: 1

Ref R1: ('key', 'identifies', 'record') -> Status: ENTAILED, Contribution: 1.0
Ref R2: ('key', 'contain', 'values') -> Status: MISSING, Contribution: 0.0


## 6. Demonstration 4: Contradiction

- **Reference Answer:** *"A primary key uniquely identifies each record in a table."*  
- **Student Answer:** *"A primary key does not uniquely identify each record in a table."*  

Stage 8 detects explicit negation, classifying the proposition as `CONTRADICTED`.  
Base score = $0.0$, and the contradiction status is explicitly preserved in diagnostics.

In [6]:
res4 = evaluate_and_score(
    ref_text="A primary key uniquely identifies each record in a table.",
    stu_text="A primary key does not uniquely identify each record in a table.",
)

print(f"Normalized Score: {res4.normalized_score} / 1.0")
print(f"Contradicted:     {res4.contradicted_count}")
print(f"Status:           {res4.proposition_results[0].status}")
print(f"Diagnostic Reasons:")
for r in res4.proposition_results[0].diagnostic_reasons:
    print(f"  - {r}")

Normalized Score: 0.0 / 1.0
Contradicted:     1
Status:           CONTRADICTED
Diagnostic Reasons:
  - Proposition is explicitly contradicted by student answer.
  - Awarded zero credit (0.00).
  - Reference proposition expresses a positive claim.
  - Student proposition contains explicit negation (not).
  - Underlying proposition terms are sufficiently aligned (triple score: 1.0000 >= 0.50).
  - Explicit polarity contradiction on structurally corresponding dependency relations.


## 7. Demonstration 5: Correct Answer + Correct Additional Information (Student B)

- **Reference Answer:** *"A primary key uniquely identifies each record in a table."*  
- **Student B:** *"A primary key uniquely identifies each record in a table and cannot contain NULL values."*  

**Base ASAG Behavior:**  
- Reference proposition `(key, identify, record)` is entailed $\rightarrow 1.0$.  
- Additional student proposition `(key, contain, values)` is preserved in `unmatched_student_relations`.  
- **No bonus is awarded:** Base score remains strictly $1.0$.

In [7]:
res_b = evaluate_and_score(
    ref_text="A primary key uniquely identifies each record in a table.",
    stu_text="A primary key uniquely identifies each record in a table and cannot contain NULL values.",
)

print(f"Normalized Score:        {res_b.normalized_score} / 1.0")
print(f"Entailed Count:          {res_b.entailed_count}")
print(f"Unmatched Student Count: {res_b.unmatched_student_count}\n")

print("Unmatched Student Information (Preserved for Stage 11-14):")
for u in res_b.unmatched_student_relations:
    print(f"  {u.to_tuple()} [Type: {u.relation_type}] (Score contribution: 0.0)")

Normalized Score:        1.0 / 1.0
Entailed Count:          1
Unmatched Student Count: 1

Unmatched Student Information (Preserved for Stage 11-14):
  ('key', 'contain', 'values') [Type: SPO] (Score contribution: 0.0)


## 8. Demonstration 6: Correct Answer + Irrelevant Additional Information (Student D)

- **Reference Answer:** *"A primary key uniquely identifies each record in a table."*  
- **Student D:** *"A primary key uniquely identifies each record in a table. SQL was developed in the 1970s."*  

**Base ASAG Behavior:**  
- Reference proposition is entailed $\rightarrow 1.0$.  
- Additional student proposition `('SQL', 'developed', None)` is preserved in `unmatched_student_relations`.  
- **No penalty is applied:** Base score remains $1.0$.

In [8]:
res_d = evaluate_and_score(
    ref_text="A primary key uniquely identifies each record in a table.",
    stu_text="A primary key uniquely identifies each record in a table. SQL was developed in the 1970s.",
)

print(f"Normalized Score:        {res_d.normalized_score} / 1.0")
print(f"Entailed Count:          {res_d.entailed_count}")
print(f"Unmatched Student Count: {res_d.unmatched_student_count}\n")

print("Unmatched Student Information:")
for u in res_d.unmatched_student_relations:
    print(f"  {u.to_tuple()} [Type: {u.relation_type}] (Score contribution: 0.0)")

Normalized Score:        1.0 / 1.0
Entailed Count:          1
Unmatched Student Count: 1

Unmatched Student Information:
  ('SQL', 'developed', None) [Type: PASSIVE_SP] (Score contribution: 0.0)


## 9. Demonstration 7: Correct Answer + Incorrect Additional Information (Student C)

- **Reference Answer:** *"A primary key uniquely identifies each record in a table."*  
- **Student C:** *"A primary key uniquely identifies each record in a table and can contain duplicate values."*  

### Critical Research Insight: The Base ASAG Limitation
Student C provides the correct definition, but introduces a major conceptual misconception (*"can contain duplicate values"*).

Because Base ASAG evaluates **only reference proposition coverage**:
- Reference proposition `(key, identify, record)` is entailed $\rightarrow 1.0$.
- Student C receives a **perfect score of $1.0 / 1.0$ ($5.0 / 5.0$)** in Base ASAG!
- Base ASAG has no mechanism to penalize the false additional proposition.

This empirical result provides the exact motivation for our **Proposed Extension (Stage 11–14)**!

In [9]:
res_c = evaluate_and_score(
    ref_text="A primary key uniquely identifies each record in a table.",
    stu_text="A primary key uniquely identifies each record in a table and can contain duplicate values.",
)

print(f"Normalized Score:        {res_c.normalized_score} / 1.0")
print(f"Scaled Score:            {res_c.scale_score(5.0)} / 5.0")
print(f"Entailed Count:          {res_c.entailed_count}")
print(f"Unmatched Student Count: {res_c.unmatched_student_count}\n")

print("Unmatched Student Misconception (Ignored by Base ASAG):")
for u in res_c.unmatched_student_relations:
    print(f"  {u.to_tuple()} [Type: {u.relation_type}]")

Normalized Score:        1.0 / 1.0
Scaled Score:            5.0 / 5.0
Entailed Count:          1
Unmatched Student Count: 1

Unmatched Student Misconception (Ignored by Base ASAG):
  ('key', 'contain', 'values') [Type: SPO]


## 10. Summary Matrix Across All Demonstrations

In [10]:
summary_table = [
    {
        "Student": "Fully Correct",
        "Ref Props": res1.total_reference_propositions,
        "Entailed": res1.entailed_count,
        "Uncertain": res1.uncertain_count,
        "Missing": res1.missing_count,
        "Contradicted": res1.contradicted_count,
        "Unmatched Student": res1.unmatched_student_count,
        "Base Score": res1.normalized_score,
        "Base 5.0 Scale": res1.scale_score(5.0),
    },
    {
        "Student": "Paraphrase (row)",
        "Ref Props": res2.total_reference_propositions,
        "Entailed": res2.entailed_count,
        "Uncertain": res2.uncertain_count,
        "Missing": res2.missing_count,
        "Contradicted": res2.contradicted_count,
        "Unmatched Student": res2.unmatched_student_count,
        "Base Score": res2.normalized_score,
        "Base 5.0 Scale": res2.scale_score(5.0),
    },
    {
        "Student": "Partial Coverage",
        "Ref Props": res3.total_reference_propositions,
        "Entailed": res3.entailed_count,
        "Uncertain": res3.uncertain_count,
        "Missing": res3.missing_count,
        "Contradicted": res3.contradicted_count,
        "Unmatched Student": res3.unmatched_student_count,
        "Base Score": res3.normalized_score,
        "Base 5.0 Scale": res3.scale_score(5.0),
    },
    {
        "Student": "Contradiction",
        "Ref Props": res4.total_reference_propositions,
        "Entailed": res4.entailed_count,
        "Uncertain": res4.uncertain_count,
        "Missing": res4.missing_count,
        "Contradicted": res4.contradicted_count,
        "Unmatched Student": res4.unmatched_student_count,
        "Base Score": res4.normalized_score,
        "Base 5.0 Scale": res4.scale_score(5.0),
    },
    {
        "Student": "Student B (Correct Extra)",
        "Ref Props": res_b.total_reference_propositions,
        "Entailed": res_b.entailed_count,
        "Uncertain": res_b.uncertain_count,
        "Missing": res_b.missing_count,
        "Contradicted": res_b.contradicted_count,
        "Unmatched Student": res_b.unmatched_student_count,
        "Base Score": res_b.normalized_score,
        "Base 5.0 Scale": res_b.scale_score(5.0),
    },
    {
        "Student": "Student D (Irrelevant Extra)",
        "Ref Props": res_d.total_reference_propositions,
        "Entailed": res_d.entailed_count,
        "Uncertain": res_d.uncertain_count,
        "Missing": res_d.missing_count,
        "Contradicted": res_d.contradicted_count,
        "Unmatched Student": res_d.unmatched_student_count,
        "Base Score": res_d.normalized_score,
        "Base 5.0 Scale": res_d.scale_score(5.0),
    },
    {
        "Student": "Student C (Incorrect Extra)",
        "Ref Props": res_c.total_reference_propositions,
        "Entailed": res_c.entailed_count,
        "Uncertain": res_c.uncertain_count,
        "Missing": res_c.missing_count,
        "Contradicted": res_c.contradicted_count,
        "Unmatched Student": res_c.unmatched_student_count,
        "Base Score": res_c.normalized_score,
        "Base 5.0 Scale": res_c.scale_score(5.0),
    },
]

pd.DataFrame(summary_table)

,Student,Ref Props,Entailed,Uncertain,Missing,Contradicted,Unmatched Student,Base Score,Base 5.0 Scale
0,Fully Correct,1,1,0,0,0,0,1.0,5.0
1,Paraphrase (row),1,0,1,0,0,0,0.5,2.5
2,Partial Coverage,2,1,0,1,0,0,0.5,2.5
3,Contradiction,1,0,0,0,1,0,0.0,0.0
4,Student B (Correct Extra),1,1,0,0,0,1,1.0,5.0
5,Student D (Irrelevant Extra),1,1,0,0,0,1,1.0,5.0
6,Student C (Incorrect Extra),1,1,0,0,0,1,1.0,5.0


## 11. Research Conclusions & Boundaries

### What Stage 9 Achieves
1. **Transparent Proposition-Coverage Scoring:** Normalized score on $[0.0, 1.0]$ directly computed from reference proposition states.
2. **Itemized Diagnostic Traceability:** For every reference proposition, Stage 9 records its status (`ENTAILED`, `UNCERTAIN`, `CONTRADICTED`, `MISSING`), its numerical contribution, and the underlying linguistic evidence.
3. **Explicit Preservation of Unmatched Information:** Unmatched student relations are tracked and preserved with zero score modification.
4. **Demonstrated Baseline Limitation:** Demonstrates that Base ASAG scores Student C as $1.0$ despite a false assertion, directly motivating the Proposed Extension in Stage 11–14.

### Strict Boundaries Maintained
- Stage 9 does **not** classify unmatched student information (reserved for Stage 13).
- Stage 9 does **not** award bonuses or penalties for extra student propositions.
- Stage 9 does **not** implement deep learning, sentence embeddings, or LLM-based grading.